# Temporalidade de status, flags e ordem dos eventos no semestre (#51)

Para não usar no modelo informação que só existe depois da saída, precisamos saber **quando cada campo é conhecido**. Este notebook responde às perguntas da issue #51 com a base v3 (`Base_academica_v3.csv`):

1. A situação do aluno (`STATUSALUNO`) descreve o semestre enquanto ele acontece ou é atualizada depois?
2. CRE, frequência e contagens de disciplinas existem no semestre da saída? O que significam nele?
3. As datas das ocorrências ficam dentro do semestre a que estão ligadas? Quando a saída é lançada no sistema?
4. Quando há mais de uma ocorrência no mesmo semestre, em que ordem elas acontecem?

Todas as saídas são agregadas. Matrícula e data de nascimento não são lidas, e nenhuma tabela mostra identificadores.

**Execução:** defina `TCC_CSV_PATH` com o caminho da `Base_academica_v3.csv` e rode as células em ordem, em kernel limpo.

In [1]:
import hashlib
import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CSV_PATH = Path(os.environ.get("TCC_CSV_PATH", "Base_academica_v3.csv"))
HASH_CSV = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest().upper()
print("Arquivo:", CSV_PATH.name)
print("SHA-256:", HASH_CSV)
HASH_V3 = "9CCE803735488354B820F9B1976612AB9D1DE5950EDB91BE57F85C415D6A2343"
if HASH_CSV != HASH_V3:
    print("Atenção: o arquivo não é a Base_academica_v3.csv registrada no relatório.")

raw = pd.read_csv(
    CSV_PATH, sep=";", encoding="utf-8-sig", dtype="string",
    usecols=lambda coluna: coluna not in {"MATRICULA", "DATANASCIMENTO"}
)
df = raw.copy()
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda valor: f"{valor:.2f}")

IDENTIFICADORES = {
    "HANDLEALUNO", "MATRICULA", "DATANASCIMENTO",
    "HANDLEALUNOANOSEMESTRE", "HANDLEALUNOCURRICULO", "HANDLECURRICULO"
}


def mostrar(titulo, tabela):
    """Exibe só tabelas sem identificadores."""
    assert not IDENTIFICADORES.intersection(map(str, tabela.columns))
    assert not IDENTIFICADORES.intersection(map(str, tabela.index.names))
    print(f"\n{titulo}")
    display(tabela)


print(f"{len(df)} linhas, {df.HANDLEALUNO.nunique()} alunos")

Arquivo: Base_academica_v4.csv
SHA-256: A37D9F2F12CB0A19DF1011BE6BA61D64E644A4C01717714DCD0B62C782DE3646
Atenção: o arquivo não é a Base_academica_v3.csv registrada no relatório.


59059 linhas, 7615 alunos


## Preparação

Semestre em ordem numérica, flags e indicadores como números, datas como datas. A conversão das datas é conferida: se algum valor não for reconhecido, a contagem aparece abaixo.

In [2]:
partes = df.ANOSEMESTRE.str.extract(r"^(\d{4})/([12])$")
assert partes.notna().all().all()
df["ANO"] = partes[0].astype(int)
df["SEMESTRE"] = partes[1].astype(int)
df["SEM"] = df.ANO * 2 + df.SEMESTRE - 1


def rotulo(sem):
    return f"{int(sem) // 2}/{int(sem) % 2 + 1}"


FLAGS = [coluna for coluna in raw.columns if coluna.startswith("TEVE")]
for coluna in FLAGS:
    df[coluna] = pd.to_numeric(raw[coluna], errors="coerce").astype("Int64")

NUMERICAS = [
    "CRENOSEMESTRE", "CREATESEMESTRE", "IA04_01", "IA04_02",
    "IA75_01", "IA75_02", "IA75_03", "IA75_04", "IA75_05", "IA34_03", "IA65_03",
    "ATIVCOMPL_MATRICULADAS", "ATIVCOMPL_APROVADAS", "ATIVCOMPL_REPROVADAS",
]
for coluna in NUMERICAS:
    df[coluna] = pd.to_numeric(raw[coluna].str.strip().str.replace(",", ".", regex=False), errors="coerce")

FORMATOS = ["%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M", "%d/%m/%Y", "%d/%m/%y %H:%M:%S", "%d/%m/%y", "%Y-%m-%d %H:%M:%S", "%Y-%m-%d"]


def converter_data(serie):
    texto = serie.str.strip()
    resultado = pd.Series(pd.NaT, index=serie.index, dtype="datetime64[ns]")
    for formato in FORMATOS:
        faltando = resultado.isna() & texto.notna() & texto.ne("")
        if not faltando.any():
            break
        resultado[faltando] = pd.to_datetime(texto[faltando], format=formato, errors="coerce")
    return resultado


DATAS = [coluna for coluna in raw.columns if coluna.startswith("DATA") and coluna != "DATANASCIMENTO"]
conversao = []
for coluna in DATAS:
    df[coluna] = converter_data(raw[coluna])
    preenchidos = raw[coluna].notna() & raw[coluna].str.strip().ne("")
    conversao.append({
        "coluna": coluna,
        "preenchidas": int(preenchidos.sum()),
        "não_reconhecidas": int((preenchidos & df[coluna].isna()).sum()),
        "com_hora": int((df[coluna].notna() & df[coluna].ne(df[coluna].dt.normalize())).sum()),
    })
mostrar("Conversão das datas", pd.DataFrame(conversao).set_index("coluna"))
assert sum(linha["não_reconhecidas"] for linha in conversao) == 0, "Há datas em formato não previsto"

SAIDA = ["TEVEDESISTENCIA", "TEVEABANDONO", "TEVECANCELAMENTO", "TEVEDESLIGAMENTO"]
df["SAIDA"] = df[SAIDA].eq(1).any(axis=1)
PRIMEIRA_SAIDA = df.loc[df.SAIDA].groupby("HANDLEALUNO").SEM.min()
df["PRIMEIRA_SAIDA"] = df.HANDLEALUNO.map(PRIMEIRA_SAIDA)
academica = df.loc[df.ORIGEMREGISTRO.eq("ACADEMICA")].copy()
print(f"{len(PRIMEIRA_SAIDA)} alunos com evento de saída; {df.SAIDA.sum()} linhas com saída")


Conversão das datas


,preenchidas,não_reconhecidas,com_hora
coluna,,,
DATADESISTENCIA,2313,0,0
DATAABANDONO,761,0,0
DATATRANCAMENTO,1727,0,0
DATACANCELAMENTO,766,0,0
DATADESLIGAMENTO,1,0,0
DATATRANSAFASTAMENTO,325,0,0
DATADIPLOMACAO,2644,0,0
DATACONCLUSAOPROGRAMA,0,0,0
DATAREINGRESSO,253,0,0


3507 alunos com evento de saída; 3841 linhas com saída


## 1. A situação do aluno (`STATUSALUNO`) é do semestre ou é atualizada depois?

**Por que conferir:** Se a situação de um semestre for atualizada depois que o aluno sai (por exemplo, virar "Inativo" quando ele não volta), ela carrega informação do futuro. Usá-la no modelo seria vazamento, mesmo no semestre anterior à saída.

**Como:** Comparamos a situação em quatro grupos de linhas acadêmicas:

- **semestre da saída:** a linha acadêmica tem evento de saída;
- **semestre anterior à saída (t−1):** o semestre imediatamente anterior à primeira saída do aluno;
- **semestres mais antigos de quem saiu:** antes de t−1;
- **alunos sem saída:** todos os semestres de quem nunca teve evento de saída.

Se a situação descreve só o semestre, t−1 deveria se parecer com os semestres mais antigos. Se ela é atualizada depois, t−1 terá mais "Inativo" ou "Desistente".

In [3]:
def grupo_status(linha):
    if pd.isna(linha.PRIMEIRA_SAIDA):
        return "4. alunos sem saída"
    if linha.SAIDA:
        return "1. semestre da saída"
    if linha.SEM == linha.PRIMEIRA_SAIDA - 1:
        return "2. semestre anterior à saída (t−1)"
    if linha.SEM < linha.PRIMEIRA_SAIDA - 1:
        return "3. semestres mais antigos de quem saiu"
    return "5. depois da primeira saída"


academica["GRUPO"] = academica.apply(grupo_status, axis=1)
status = pd.crosstab(academica.GRUPO, academica.STATUSALUNO.fillna("(vazio)"))
status_pct = pd.crosstab(academica.GRUPO, academica.STATUSALUNO.fillna("(vazio)"), normalize="index") * 100
mostrar("Linhas acadêmicas por grupo e situação do aluno (contagem)", status.assign(total=status.sum(axis=1)))
mostrar("Linhas acadêmicas por grupo e situação do aluno (% da linha)", status_pct)


Linhas acadêmicas por grupo e situação do aluno (contagem)


STATUSALUNO,Ativo,Desconhecido,Em Mobilidade Acadêmica,Inativo,Matrícula Pendente,total
GRUPO,,,,,,
1. semestre da saída,1,0,0,1487,0,1488
2. semestre anterior à saída (t−1),1258,15,0,212,4,1489
3. semestres mais antigos de quem saiu,4958,57,0,115,3,5133
4. alunos sem saída,33736,172,3,211,9,34131
5. depois da primeira saída,2124,5,0,45,2,2176



Linhas acadêmicas por grupo e situação do aluno (% da linha)


STATUSALUNO,Ativo,Desconhecido,Em Mobilidade Acadêmica,Inativo,Matrícula Pendente
GRUPO,,,,,
1. semestre da saída,0.07,0.00,0.00,99.93,0.00
2. semestre anterior à saída (t−1),84.49,1.01,0.00,14.24,0.27
3. semestres mais antigos de quem saiu,96.59,1.11,0.00,2.24,0.06
4. alunos sem saída,98.84,0.50,0.01,0.62,0.03
5. depois da primeira saída,97.61,0.23,0.00,2.07,0.09


**Segunda forma de olhar:** Se a situação de um semestre "prevê" com muita precisão se o aluno terá matrícula no semestre seguinte, ela provavelmente foi registrada depois de se saber o que aconteceu. A tabela mostra, para cada situação, a porcentagem de linhas cujo aluno tem matrícula efetivada no semestre seguinte. Ficam de fora as linhas do último semestre da base, que não têm semestre seguinte observado.

In [4]:
matricula_por_semestre = set(zip(academica.HANDLEALUNO, academica.SEM))
observaveis = academica.loc[academica.SEM.lt(df.SEM.max())].copy()
observaveis["MATRICULA_SEGUINTE"] = [
    (aluno, sem + 1) in matricula_por_semestre
    for aluno, sem in zip(observaveis.HANDLEALUNO, observaveis.SEM)
]
seguinte = observaveis.groupby(observaveis.STATUSALUNO.fillna("(vazio)")).agg(
    linhas=("MATRICULA_SEGUINTE", "size"),
    com_matricula_seguinte_pct=("MATRICULA_SEGUINTE", lambda serie: serie.mean() * 100),
).sort_values("linhas", ascending=False)
mostrar("Situação do aluno × matrícula efetivada no semestre seguinte", seguinte)


Situação do aluno × matrícula efetivada no semestre seguinte


,linhas,com_matricula_seguinte_pct
STATUSALUNO,,
Ativo,40680,88.69
Inativo,2007,10.96
Desconhecido,249,94.78
Matrícula Pendente,16,56.25
Em Mobilidade Acadêmica,1,100.00


## 2. CRE, frequência e disciplinas no semestre da saída

**Por que conferir:** Os indicadores de desempenho de um semestre só ficam completos no fim dele. Se a saída acontece no meio do semestre, os valores desse semestre podem estar vazios, zerados ou incompletos por causa da própria saída.

**Como:** Para cada indicador, mostramos a porcentagem de linhas preenchidas, a porcentagem de zeros entre as preenchidas e a média, nos mesmos grupos da seção 1. As linhas que vêm só de ocorrência já não têm esses indicadores (ficam vazias), então a tabela usa só linhas acadêmicas.

In [5]:
INDICADORES_SEMESTRE = {
    "CRENOSEMESTRE": "CRE no semestre",
    "CREATESEMESTRE": "CRE acumulado",
    "IA04_01": "Frequência (%)",
    "IA04_02": "Disciplinas com >25% de faltas",
    "IA75_04": "Disciplinas matriculadas",
    "IA75_03": "Disciplinas aprovadas",
    "IA34_03": "Disciplinas reprovadas",
    "IA65_03": "Disciplinas retiradas/canceladas",
    "ATIVCOMPL_MATRICULADAS": "Ativ. complementares matriculadas",
}
linhas = []
for grupo, dados in academica.groupby("GRUPO"):
    for coluna, nome in INDICADORES_SEMESTRE.items():
        valores = dados[coluna]
        preenchidos = valores.dropna()
        linhas.append({
            "grupo": grupo,
            "indicador": nome,
            "linhas": len(dados),
            "preenchidas_pct": valores.notna().mean() * 100,
            "zeros_pct": preenchidos.eq(0).mean() * 100 if len(preenchidos) else np.nan,
            "media": preenchidos.mean() if len(preenchidos) else np.nan,
        })
indicadores = pd.DataFrame(linhas)
indicadores["grupo"] = indicadores.grupo.str.split(". ", n=1).str[1]
ORDEM_GRUPOS = [
    "semestre da saída", "semestre anterior à saída (t−1)",
    "semestres mais antigos de quem saiu", "alunos sem saída", "depois da primeira saída",
]
ORDEM_GRUPOS = [grupo for grupo in ORDEM_GRUPOS if grupo in set(indicadores.grupo)]
for medida, titulo in [
    ("preenchidas_pct", "% de linhas com o indicador preenchido"),
    ("zeros_pct", "% de zeros entre as linhas preenchidas"),
    ("media", "Média entre as linhas preenchidas"),
]:
    mostrar(
        titulo,
        indicadores.pivot(index="indicador", columns="grupo", values=medida)
        .reindex(index=list(INDICADORES_SEMESTRE.values()), columns=ORDEM_GRUPOS)
    )


% de linhas com o indicador preenchido


grupo,semestre da saída,semestre anterior à saída (t−1),semestres mais antigos de quem saiu,alunos sem saída,depois da primeira saída
indicador,,,,,
CRE no semestre,4.30,86.23,97.86,99.46,98.02
CRE acumulado,47.04,90.46,98.62,99.68,98.53
Frequência (%),0.00,85.56,97.45,99.26,97.66
Disciplinas com >25% de faltas,0.00,85.56,97.45,99.26,97.66
Disciplinas matriculadas,100.00,100.00,100.00,100.00,100.00
Disciplinas aprovadas,100.00,100.00,100.00,100.00,100.00
Disciplinas reprovadas,100.00,100.00,100.00,100.00,100.00
Disciplinas retiradas/canceladas,100.00,100.00,100.00,100.00,100.00
Ativ. complementares matriculadas,100.00,100.00,100.00,100.00,100.00



% de zeros entre as linhas preenchidas


grupo,semestre da saída,semestre anterior à saída (t−1),semestres mais antigos de quem saiu,alunos sem saída,depois da primeira saída
indicador,,,,,
CRE no semestre,35.94,20.79,6.89,1.32,5.06
CRE acumulado,77.14,8.31,1.24,0.17,1.12
Frequência (%),NaN,0.00,0.00,0.00,0.00
Disciplinas com >25% de faltas,NaN,55.89,70.77,91.14,81.88
Disciplinas matriculadas,0.67,0.27,0.97,0.78,0.00
Disciplinas aprovadas,98.45,58.90,27.78,5.33,13.10
Disciplinas reprovadas,100.00,21.29,20.69,57.13,46.42
Disciplinas retiradas/canceladas,93.08,77.70,72.59,81.32,75.00
Ativ. complementares matriculadas,18.35,30.09,31.79,24.85,34.97



Média entre as linhas preenchidas


grupo,semestre da saída,semestre anterior à saída (t−1),semestres mais antigos de quem saiu,alunos sem saída,depois da primeira saída
indicador,,,,,
CRE no semestre,16.11,23.80,37.58,62.49,54.91
CRE acumulado,8.76,30.10,41.54,61.82,56.36
Frequência (%),NaN,79.33,85.18,92.28,88.95
Disciplinas com >25% de faltas,NaN,1.17,0.60,0.15,0.40
Disciplinas matriculadas,4.67,4.85,5.04,5.76,5.71
Disciplinas aprovadas,0.03,0.94,2.09,4.28,3.64
Disciplinas reprovadas,0.00,2.79,2.27,0.92,1.33
Disciplinas retiradas/canceladas,0.15,0.41,0.52,0.32,0.47
Ativ. complementares matriculadas,1.07,1.00,1.01,1.21,1.07


**Semestre da saída: o aluno chegou a cursar?** Entre as linhas acadêmicas com saída, separamos as que não têm nenhuma disciplina matriculada, as que têm disciplinas mas CRE vazio ou zero, e as que têm CRE positivo.

In [6]:
saida_academica = academica.loc[academica.GRUPO.eq("1. semestre da saída")]
situacao_saida = np.select(
    [
        saida_academica.IA75_04.fillna(0).eq(0),
        saida_academica.CRENOSEMESTRE.isna() | saida_academica.CRENOSEMESTRE.eq(0),
    ],
    ["sem disciplina matriculada", "com disciplinas, CRE vazio ou zero"],
    default="com disciplinas e CRE positivo",
)
mostrar(
    "Linhas acadêmicas com saída: o semestre foi cursado?",
    pd.Series(situacao_saida).value_counts().rename_axis("situação").to_frame("linhas")
    .assign(pct=lambda tabela: tabela.linhas / tabela.linhas.sum() * 100)
)


Linhas acadêmicas com saída: o semestre foi cursado?


,linhas,pct
situação,,
"com disciplinas, CRE vazio ou zero",1437,96.57
com disciplinas e CRE positivo,41,2.76
sem disciplina matriculada,10,0.67


## 3. Datas das ocorrências

**Por que conferir.** As flags dizem que uma ocorrência está ligada a um semestre, mas não quando ela aconteceu. A data permite ordenar eventos, desde que fique claro o que ela representa.

**3.1 Lançamento da saída no sistema.** Diferença entre a data em que a saída foi lançada e a data registrada para a saída, nas linhas da base (só graduação).

In [7]:
com_saida = df.loc[df.SAIDA & df.DATAPRIMEIRASAIDA.notna() & df.DATAINCLUSAOSAIDA.notna()]
dias = (com_saida.DATAINCLUSAOSAIDA.dt.normalize() - com_saida.DATAPRIMEIRASAIDA.dt.normalize()).dt.days
faixas = pd.cut(dias, [-np.inf, -1, 0, 30, 180, np.inf],
                labels=["inclusão antes da ocorrência", "mesmo dia", "1 a 30 dias depois", "31 a 180 dias depois", "mais de 180 dias depois"])
mostrar(
    "Intervalo entre a data da saída e o lançamento no sistema",
    faixas.value_counts(sort=False).rename_axis("faixa").to_frame("linhas")
    .assign(pct=lambda tabela: tabela.linhas / tabela.linhas.sum() * 100)
)


Intervalo entre a data da saída e o lançamento no sistema


,linhas,pct
faixa,,
inclusão antes da ocorrência,2,0.05
mesmo dia,1805,46.99
1 a 30 dias depois,1919,49.96
31 a 180 dias depois,105,2.73
mais de 180 dias depois,10,0.26


**3.2 A data cai na metade do ano do semestre?** Para cada tipo de ocorrência, comparamos o mês da data com a metade do ano do semestre a que ela está ligada (janeiro a junho para o 1º semestre, julho a dezembro para o 2º). É uma aproximação: o calendário de aulas não está na base.

In [8]:
EVENTOS = {
    "Desistência": ("TEVEDESISTENCIA", "DATADESISTENCIA"),
    "Abandono": ("TEVEABANDONO", "DATAABANDONO"),
    "Cancelamento": ("TEVECANCELAMENTO", "DATACANCELAMENTO"),
    "Desligamento": ("TEVEDESLIGAMENTO", "DATADESLIGAMENTO"),
    "Trancamento": ("TEVETRANCAMENTO", "DATATRANCAMENTO"),
    "Transferência/afastamento": ("TEVETRANSAFASTAMENTO", "DATATRANSAFASTAMENTO"),
    "Afastamento sem ocorrência": ("TEVEAFASTAMENTOSEMOCORRENCIA", "DATAAFASTAMENTOSEMOCORRENCIA"),
    "Reingresso": ("TEVEREINGRESSO", "DATAREINGRESSO"),
    "Rematrícula": ("TEVEREMATRICULA", "DATAREMATRICULA"),
    "Mudança de curso": ("TEVEMUDANCACURSO", "DATAMUDANCACURSO"),
    "Migração de currículo": ("TEVEMIGRACAOCURRICULO", "DATAMIGRACAOCURRICULO"),
    "Diplomação": ("TEVEDIPLOMACAO", "DATADIPLOMACAO"),
    "Prazo esgotado": ("TEVEPRAZOESGOTADO", "DATAPRAZOESGOTADO"),
    "Falecimento": ("TEVEFALECIMENTO", "DATAFALECIMENTO"),
}


def posicao_no_ano(datas, ano, semestre):
    inicio = np.where(semestre.eq(1), 1, 7)
    fim = np.where(semestre.eq(1), 6, 12)
    indice_data = datas.dt.year * 12 + datas.dt.month
    indice_inicio = ano * 12 + inicio
    indice_fim = ano * 12 + fim
    return np.select(
        [datas.isna(), indice_data < indice_inicio, indice_data > indice_fim],
        ["sem data", "antes", "depois"],
        default="dentro",
    )


linhas = []
for nome, (flag, data) in EVENTOS.items():
    ocorre = df.loc[df[flag].eq(1)]
    if ocorre.empty:
        continue
    posicao = pd.Series(posicao_no_ano(ocorre[data], ocorre.ANO, ocorre.SEMESTRE))
    contagem = posicao.value_counts()
    linhas.append({
        "ocorrência": nome,
        "linhas": len(ocorre),
        **{f"{rotulo_pos}_pct": contagem.get(rotulo_pos, 0) / len(ocorre) * 100
           for rotulo_pos in ["antes", "dentro", "depois", "sem data"]},
    })
mostrar("Data da ocorrência em relação à metade do ano do semestre", pd.DataFrame(linhas).set_index("ocorrência"))


Data da ocorrência em relação à metade do ano do semestre


,linhas,antes_pct,dentro_pct,depois_pct,sem data_pct
ocorrência,,,,,
Desistência,2313,3.46,95.68,0.86,0.00
Abandono,761,0.00,100.00,0.00,0.00
Cancelamento,766,12.92,86.81,0.26,0.00
Desligamento,1,0.00,100.00,0.00,0.00
Trancamento,1727,7.53,92.30,0.17,0.00
Transferência/afastamento,325,5.54,93.85,0.62,0.00
Afastamento sem ocorrência,331,0.60,7.55,91.84,0.00
Reingresso,253,12.25,87.75,0.00,0.00
Rematrícula,593,17.20,82.63,0.17,0.00


## 4. Ordem dos eventos no mesmo semestre

**Por que conferir.** Quando um aluno tem mais de uma ocorrência no mesmo semestre (por exemplo, trancamento e cancelamento, ou desistência e reingresso), só as flags não dizem qual veio primeiro.

**Como.** Juntamos as ocorrências de cada aluno por semestre, inclusive de currículos diferentes, e pegamos a primeira data de cada tipo. As quatro saídas formam o grupo "Saída". Para cada par de tipos que aparecem juntos no mesmo semestre, contamos em quantos semestres o primeiro veio antes, no mesmo dia ou depois do segundo. A data não tem hora: eventos no mesmo dia não podem ser ordenados.

In [9]:
GRUPOS_ORDEM = {
    "Saída": SAIDA,
    "Trancamento": ["TEVETRANCAMENTO"],
    "Transferência/afastamento": ["TEVETRANSAFASTAMENTO"],
    "Afastamento sem ocorrência": ["TEVEAFASTAMENTOSEMOCORRENCIA"],
    "Reingresso": ["TEVEREINGRESSO"],
    "Rematrícula": ["TEVEREMATRICULA"],
    "Mudança de curso": ["TEVEMUDANCACURSO"],
    "Migração de currículo": ["TEVEMIGRACAOCURRICULO"],
    "Diplomação": ["TEVEDIPLOMACAO"],
    "Prazo esgotado": ["TEVEPRAZOESGOTADO"],
}
DATA_DA_FLAG = {flag: data for flag, data in EVENTOS.values()}

datas_grupo = pd.DataFrame({"HANDLEALUNO": df.HANDLEALUNO, "SEM": df.SEM})
for grupo, flags in GRUPOS_ORDEM.items():
    datas_flags = pd.concat(
        [df[DATA_DA_FLAG[flag]].where(df[flag].eq(1)) for flag in flags], axis=1
    )
    datas_grupo[grupo] = datas_flags.min(axis=1)
    datas_grupo[grupo + "_flag"] = df[flags].eq(1).any(axis=1)

por_semestre = datas_grupo.groupby(["HANDLEALUNO", "SEM"]).agg(
    **{grupo: (grupo, "min") for grupo in GRUPOS_ORDEM},
    **{grupo + "_flag": (grupo + "_flag", "max") for grupo in GRUPOS_ORDEM},
)

nomes = list(GRUPOS_ORDEM)
linhas = []
for i, primeiro in enumerate(nomes):
    for segundo in nomes[i + 1:]:
        juntos = por_semestre.loc[por_semestre[primeiro + "_flag"] & por_semestre[segundo + "_flag"]]
        if juntos.empty:
            continue
        a, b = juntos[primeiro], juntos[segundo]
        linhas.append({
            "evento A": primeiro,
            "evento B": segundo,
            "aluno_semestres": len(juntos),
            "A antes de B": int((a.dt.normalize() < b.dt.normalize()).sum()),
            "mesmo dia": int((a.dt.normalize() == b.dt.normalize()).sum()),
            "A depois de B": int((a.dt.normalize() > b.dt.normalize()).sum()),
            "sem data": int((a.isna() | b.isna()).sum()),
        })
ordem = pd.DataFrame(linhas).sort_values("aluno_semestres", ascending=False).set_index(["evento A", "evento B"])
mostrar("Pares de eventos no mesmo aluno/semestre e sua ordem", ordem)


Pares de eventos no mesmo aluno/semestre e sua ordem


,,aluno_semestres,A antes de B,mesmo dia,A depois de B,sem data
evento A,evento B,,,,,
Saída,Migração de currículo,136,2,0,134,0
Trancamento,Migração de currículo,123,4,1,118,0
Migração de currículo,Diplomação,122,122,0,0,0
Rematrícula,Migração de currículo,68,37,31,0,0
Reingresso,Migração de currículo,46,22,24,0,0
Transferência/afastamento,Migração de currículo,34,1,0,33,0
Saída,Rematrícula,23,0,0,23,0
Rematrícula,Mudança de curso,22,18,4,0,0
Trancamento,Rematrícula,21,0,0,21,0


**Mais de um tipo de saída no mesmo semestre.** Também conferimos se as quatro saídas aparecem juntas no mesmo aluno/semestre e em que ordem.

In [10]:
NOMES_SAIDA = {
    "TEVEDESISTENCIA": "Desistência", "TEVEABANDONO": "Abandono",
    "TEVECANCELAMENTO": "Cancelamento", "TEVEDESLIGAMENTO": "Desligamento",
}
datas_saida = pd.DataFrame({"HANDLEALUNO": df.HANDLEALUNO, "SEM": df.SEM})
for flag, nome in NOMES_SAIDA.items():
    datas_saida[nome] = df[DATA_DA_FLAG[flag]].where(df[flag].eq(1))
saida_semestre = datas_saida.groupby(["HANDLEALUNO", "SEM"]).min()
tipos_por_semestre = saida_semestre.notna().sum(axis=1)
mostrar(
    "Quantos tipos de saída aparecem no mesmo aluno/semestre",
    tipos_por_semestre[tipos_por_semestre.gt(0)].value_counts().sort_index()
    .rename_axis("tipos de saída").to_frame("aluno_semestres")
)
linhas = []
nomes_saida = list(NOMES_SAIDA.values())
for i, primeiro in enumerate(nomes_saida):
    for segundo in nomes_saida[i + 1:]:
        juntos = saida_semestre.loc[saida_semestre[primeiro].notna() & saida_semestre[segundo].notna()]
        if juntos.empty:
            continue
        a, b = juntos[primeiro].dt.normalize(), juntos[segundo].dt.normalize()
        linhas.append({"evento A": primeiro, "evento B": segundo, "aluno_semestres": len(juntos),
                       "A antes de B": int((a < b).sum()), "mesmo dia": int((a == b).sum()),
                       "A depois de B": int((a > b).sum())})
if linhas:
    mostrar("Ordem entre tipos de saída no mesmo semestre", pd.DataFrame(linhas).set_index(["evento A", "evento B"]))
else:
    print("Nenhum aluno/semestre com dois tipos de saída.")


Quantos tipos de saída aparecem no mesmo aluno/semestre


,aluno_semestres
tipos de saída,
1,3723
2,5



Ordem entre tipos de saída no mesmo semestre


,,aluno_semestres,A antes de B,mesmo dia,A depois de B
evento A,evento B,,,,
Desistência,Cancelamento,5,0,0,5


## 5. Resumo para o relatório

As tabelas acima alimentam a tabela de temporalidade dos campos (`relatorio_temporalidade_campos.md`). Esta célula só confirma que a base não mudou durante a execução.

In [11]:
assert hashlib.sha256(CSV_PATH.read_bytes()).hexdigest().upper() == HASH_CSV
print("Base inalterada durante a execução:", CSV_PATH.name)

Base inalterada durante a execução: Base_academica_v4.csv
